#Question-6


###Long-Context Modeling: From RNNs to Selective State Space Models

In this section we analyze long-context sequence modeling using multiple architectures.

We construct a modified dataset called **IMDb-Long** by inserting neutral text from WikiText-2 between parts of IMDb reviews. This prevents label leakage and forces models to handle long sequences.

We compare the following models:

• LSTM  
• Transformer Encoder  
• Linear State Space Model (SSM)  
• Selective State Space Model

Experiments are performed for sequence lengths:

- 256 tokens
- 512 tokens
- 1024 tokens

Performance is evaluated using:

• classification accuracy  
• training time per epoch

###Import Required Libraries

The following libraries are used in the implementation:

- **PyTorch** for building and training neural networks  
- **NumPy** for numerical computation  
- **HuggingFace datasets** for loading IMDb and WikiText-2 datasets  
- **Matplotlib** for plotting experimental results


In [2]:
import time
import random
import re
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset
from collections import Counter

## Reproducibility

To ensure reproducible results, we fix the random seed for:

- Python random module
- NumPy
- PyTorch

This ensures that experiments produce consistent results across runs.

In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


## Loading IMDb and WikiText-2 Datasets

Two datasets are used:

### IMDb Dataset
The IMDb dataset contains movie reviews labeled with sentiment:
- 0 → Negative review
- 1 → Positive review

### WikiText-2 Dataset
WikiText-2 contains neutral Wikipedia text. This dataset is used as **neutral filler text** in the IMDb-Long dataset.

Using Wikipedia text ensures that filler sentences do not leak sentiment information.

In [4]:
imdb = load_dataset("imdb")
wiki = load_dataset("wikitext", "wikitext-2-raw-v1")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

...

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

## Tokenization

Text data must be converted into tokens before it can be processed by neural networks.

We apply a simple tokenizer that:

1. Converts text to lowercase
2. Extracts word tokens using regular expressions

The same tokenizer is used across all models to maintain consistency.

In [5]:
def simple_tokenize(text):
    text = text.lower()
    tokens = re.findall(r"\b\w+\b", text)
    return tokens

## Extracting Neutral Sentences from WikiText-2

Neutral sentences are extracted from the WikiText-2 dataset.

These sentences will later be inserted into IMDb reviews to construct the **IMDb-Long dataset**.

Using neutral Wikipedia text ensures:

- minimal sentiment bias
- independence from IMDb labels
- realistic long-context sequences

In [6]:
def split_into_sentences(text):
    text = text.strip()
    if not text:
        return []
    sentences = re.split(r'[.!?]+', text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 5]
    return sentences

neutral_sentences = []

for item in wiki["train"]:
    txt = item["text"].strip()
    if len(txt) == 0:
        continue
    sents = split_into_sentences(txt)
    neutral_sentences.extend(sents)

print("Number of neutral sentences:", len(neutral_sentences))
print(neutral_sentences[:5])

Number of neutral sentences: 89709
['= Valkyria Chronicles III =', 'Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit', 'Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media', 'Vision for the PlayStation Portable', 'Released in January 2011 in Japan , it is the third game in the Valkyria series']


## Dataset Subset

Training very long sequences can be computationally expensive.

Therefore we use a subset of the IMDb dataset:

Training samples: 3000  
Test samples: 1000

This allows experiments to run within a reasonable time while still demonstrating long-context behavior.

In [7]:
train_subset_size = 3000
test_subset_size = 1000

train_data_raw = imdb["train"].shuffle(seed=42).select(range(train_subset_size))
test_data_raw = imdb["test"].shuffle(seed=42).select(range(test_subset_size))

## Constructing the IMDb-Long Dataset

Each IMDb review is modified to create a long-context sequence.

Steps:

1. Tokenize the review
2. Split review into three parts:

$$
review = (P_1, P_2, P_3)
$$

3. Remove the middle section \(P_2\)
4. Insert neutral filler sentences from WikiText-2
5. Construct final sequence:

$$
final = P_1 \; || \; neutral\ filler \; || \; P_3
$$

Additional neutral sentences are appended if necessary to reach the target sequence length.

This forces models to capture long-range dependencies.

In [8]:
def build_imdb_long(review_text, target_len, neutral_sentences, rng):
    review_tokens = simple_tokenize(review_text)

    if len(review_tokens) < 6:
        review_tokens = review_tokens + ["movie"] * (6 - len(review_tokens))

    n = len(review_tokens)
    p1_end = n // 3
    p2_end = 2 * n // 3

    P1 = review_tokens[:p1_end]
    P3 = review_tokens[p2_end:]

    final_tokens = P1.copy()

    while len(final_tokens) + len(P3) < target_len:
        sent = rng.choice(neutral_sentences)
        sent_tokens = simple_tokenize(sent)
        final_tokens.extend(sent_tokens)

    final_tokens.extend(P3)
    final_tokens = final_tokens[:target_len]

    if len(final_tokens) < target_len:
        final_tokens += ["pad"] * (target_len - len(final_tokens))

    return final_tokens

##9

In [9]:
def build_vocab(dataset_list, max_vocab=20000, min_freq=2):
    counter = Counter()
    for tokens, _ in dataset_list:
        counter.update(tokens)

    vocab = {"<pad>": 0, "<unk>": 1}
    for word, freq in counter.most_common(max_vocab):
        if freq >= min_freq and word not in vocab:
            vocab[word] = len(vocab)
    return vocab

In [10]:
def prepare_imdb_long_dataset(raw_data, target_len, neutral_sentences, seed=42):
    rng = random.Random(seed)
    processed = []

    for item in raw_data:
        tokens = build_imdb_long(item["text"], target_len, neutral_sentences, rng)
        label = item["label"]
        processed.append((tokens, label))

    return processed

In [11]:
def numericalize_dataset(dataset_list, vocab):
    result = []
    unk = vocab["<unk>"]
    for tokens, label in dataset_list:
        ids = [vocab.get(tok, unk) for tok in tokens]
        result.append((ids, label))
    return result

In [12]:
class TextDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x, y = self.data[idx]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

In [13]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=2048):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).float().unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)

        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

## Model Architectures

We implement four sequence models for comparison.

### 1. LSTM
A recurrent neural network designed to handle long-term dependencies using memory cells and gates.

### 2. Transformer Encoder
A self-attention based architecture that allows each token to attend to all other tokens in the sequence.

### 3. Linear State Space Model (SSM)
A linear dynamical system defined by the recurrence:

$$
h_t = A h_{t-1} + B x_t
$$

$$
y_t = C h_t
$$

### 4. Selective State Space Model
An extension of the linear SSM that introduces input-dependent gating:

$$
g_t = \sigma(W_g x_t)
$$

$$
y_t = g_t \odot (C h_t)
$$

The gating mechanism allows the model to selectively control information flow.

In [14]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        emb = self.embedding(x)
        _, (h_n, _) = self.lstm(emb)
        out = self.fc(h_n[-1])
        return out

In [15]:
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, hidden_dim, num_layers, num_classes, max_len=2048):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_enc = PositionalEncoding(embed_dim, max_len=max_len)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        emb = self.embedding(x)
        emb = self.pos_enc(emb)
        h = self.encoder(emb)
        pooled = h.mean(dim=1)
        out = self.fc(pooled)
        return out

In [16]:
class LinearSSM(nn.Module):
    def __init__(self, d_in, d_hidden, num_classes):
        super().__init__()
        self.A = nn.Parameter(torch.randn(d_hidden, d_hidden) * 0.1)
        self.B = nn.Linear(d_in, d_hidden)
        self.C = nn.Linear(d_hidden, d_hidden)
        self.fc = nn.Linear(d_hidden, num_classes)

    def forward(self, x):
        batch, seq_len, _ = x.shape

        # initialize hidden state
        h = torch.zeros(batch, self.A.size(0), device=x.device)

        for t in range(seq_len):
            x_t = x[:, t]

            # recurrence: h = A h + B x_t
            h = torch.matmul(h, self.A.T) + self.B(x_t)

        # output
        y = self.C(h)
        out = self.fc(y)
        return out

In [17]:
class SelectiveSSM(nn.Module):
    def __init__(self, d_in, d_hidden, num_classes):
        super().__init__()
        self.A = nn.Parameter(torch.randn(d_hidden, d_hidden) * 0.1)
        self.B = nn.Linear(d_in, d_hidden)
        self.C = nn.Linear(d_hidden, d_hidden)

        # gating network
        self.W_g = nn.Linear(d_in, d_hidden)

        self.fc = nn.Linear(d_hidden, num_classes)

    def forward(self, x):
        batch, seq_len, _ = x.shape

        # initialize hidden state
        h = torch.zeros(batch, self.A.size(0), device=x.device)

        for t in range(seq_len):
            x_t = x[:, t]

            # update state
            h = torch.matmul(h, self.A.T) + self.B(x_t)

            # compute gate
            g = torch.sigmoid(self.W_g(x_t))

            # selective output
            y = g * self.C(h)

        # classification
        out = self.fc(y)
        return out

In [18]:
class EmbeddedLinearSSM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.ssm = LinearSSM(embed_dim, hidden_dim, num_classes)

    def forward(self, x):
        emb = self.embedding(x)
        return self.ssm(emb)

In [19]:
class EmbeddedSelectiveSSM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.ssm = SelectiveSSM(embed_dim, hidden_dim, num_classes)

    def forward(self, x):
        emb = self.embedding(x)
        return self.ssm(emb)

In [20]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    start_time = time.time()

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    epoch_time = time.time() - start_time
    avg_loss = total_loss / total
    acc = correct / total

    return avg_loss, acc, epoch_time

In [21]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)

        total_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    avg_loss = total_loss / total
    acc = correct / total
    return avg_loss, acc

In [22]:
def run_experiment(seq_len, epochs=3, batch_size=32, embed_dim=128, hidden_dim=128):
    print(f"\n===== Sequence Length = {seq_len} =====")

    train_processed = prepare_imdb_long_dataset(train_data_raw, seq_len, neutral_sentences, seed=42)
    test_processed = prepare_imdb_long_dataset(test_data_raw, seq_len, neutral_sentences, seed=123)

    vocab = build_vocab(train_processed, max_vocab=20000, min_freq=2)
    train_num = numericalize_dataset(train_processed, vocab)
    test_num = numericalize_dataset(test_processed, vocab)

    train_ds = TextDataset(train_num)
    test_ds = TextDataset(test_num)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    models = {
        "LSTM": LSTMClassifier(len(vocab), embed_dim, hidden_dim, 2).to(device),
        "Transformer": TransformerClassifier(len(vocab), embed_dim, num_heads=4, hidden_dim=256, num_layers=2, num_classes=2, max_len=seq_len).to(device),
        "LinearSSM": EmbeddedLinearSSM(len(vocab), embed_dim, hidden_dim, 2).to(device),
        "SelectiveSSM": EmbeddedSelectiveSSM(len(vocab), embed_dim, hidden_dim, 2).to(device)
    }

    criterion = nn.CrossEntropyLoss()
    results = []

    for model_name, model in models.items():
        print(f"\nTraining {model_name}...")
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

        train_times = []
        test_acc = 0.0

        for epoch in range(epochs):
            train_loss, train_acc, epoch_time = train_one_epoch(model, train_loader, optimizer, criterion, device)
            val_loss, test_acc = evaluate(model, test_loader, criterion, device)
            train_times.append(epoch_time)

            print(f"Epoch {epoch+1}: "
                  f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, "
                  f"test_acc={test_acc:.4f}, time={epoch_time:.2f}s")

        results.append({
            "seq_len": seq_len,
            "model": model_name,
            "accuracy": test_acc,
            "train_time_per_epoch": np.mean(train_times)
        })

    return results


Let:

\(L\) = sequence length  
\(d\) = hidden dimension

### Transformer Complexity

Self-attention requires computing:

$$
Attention(Q,K,V) = softmax\left(\frac{QK^T}{\sqrt{d}}\right)V
$$

Computing $$QK^T$$requires:

$$
O(L^2 d)
$$

Therefore Transformer complexity per layer is:

$$
O(L^2 d)
$$

---

### RNN Complexity

For an RNN:

$$
h_t = W_h h_{t-1} + W_x x_t
$$

Each step costs:

$$
O(d^2)
$$

For sequence length \(L\):

$$
O(L d^2)
$$

---

### Ratio

$$
\frac{Transformer}{RNN}
=
\frac{L^2 d}{L d^2}
=
\frac{L}{d}
$$

For:

$$
L = 1024, \quad d = 256
$$

$$
\frac{1024}{256} = 4
$$

Thus Transformer computation is approximately **4× more expensive than RNN** for these parameters.

## Gradient Propagation in RNNs

Given recurrence:

$$
h_t = A h_{t-1} + B x_t
$$

Expanding recursively:

$$
h_t = A^t h_0 + \sum_{i=1}^{t} A^{t-i} B x_i
$$

---

### Gradient

$$
\frac{\partial h_t}{\partial h_{t-k}} = A^k
$$

---

### Spectral Radius Analysis

Let $$ \rho(A) $$ be the spectral radius.

If:

$$
\rho(A) > 1
$$

then gradients grow exponentially → **exploding gradients**

If:

$$
\rho(A) < 1
$$

then gradients decay exponentially → **vanishing gradients**

## State Space Models

Discrete state space system:

$$
h_k = A h_{k-1} + B x_k
$$

$$
y_k = C h_k
$$

Substituting recursively:

$$
y_k = \sum_{i=0}^{k} C A^{k-i} B x_i
$$

Define kernel:

$$
K_{k-i} = C A^{k-i} B
$$

Then:

$$
y_k = \sum_{i=0}^{k} K_{k-i} x_i
$$

This is equivalent to a **1D convolution** over the input sequence.

The system is **time-invariant** because matrices \(A,B,C\) remain constant over time.

In [ ]:
all_results = []

for seq_len in [256, 512, 1024]:
    res = run_experiment(seq_len, epochs=3, batch_size=32, embed_dim=128, hidden_dim=128)
    all_results.extend(res)

results_df = pd.DataFrame(all_results)
results_df


===== Sequence Length = 256 =====

Training LSTM...
Epoch 1: train_loss=0.6939, train_acc=0.5227, test_acc=0.5290, time=266.73s
Epoch 2: train_loss=0.6392, train_acc=0.6433, test_acc=0.5820, time=77.81s
Epoch 3: train_loss=0.5321, train_acc=0.7373, test_acc=0.5850, time=24.32s

Training Transformer...
Epoch 1: train_loss=0.7114, train_acc=0.5087, test_acc=0.5160, time=139.84s
Epoch 2: train_loss=0.6300, train_acc=0.6347, test_acc=0.6710, time=138.90s
Epoch 3: train_loss=0.5479, train_acc=0.7210, test_acc=0.7210, time=138.14s

Training LinearSSM...
Epoch 1: train_loss=58203513605419872.0000, train_acc=0.4980, test_acc=0.5060, time=44.07s
Epoch 2: train_loss=19798025487166.1211, train_acc=0.5093, test_acc=0.4950, time=40.57s
Epoch 3: train_loss=9761949965352.9609, train_acc=0.5200, test_acc=0.4940, time=43.29s

Training SelectiveSSM...
Epoch 1: train_loss=29638558325045.2461, train_acc=0.5110, test_acc=0.4860, time=43.04s
Epoch 2: train_loss=3024156521.8133, train_acc=0.5247, test_acc=0

## Selective State Space Models

Selective SSM introduces an input-dependent gating mechanism.

State update:

$$
h_k = A h_{k-1} + B x_k
$$

Gate computation:

$$
g_k = \sigma(W_g x_k)
$$

Output:

$$
y_k = g_k \odot (C h_k)
$$

Substituting expanded form:

$$
y_k = g_k \odot \sum_{i=0}^{k} C A^{k-i} B x_i
$$

Because $$g_k$$depends on input $$x_k$$, the system is **no longer time-invariant**.

This also means the model cannot be represented as a fixed convolution kernel.

In [ ]:
plt.figure(figsize=(8, 5))

for model_name in results_df["model"].unique():
    subset = results_df[results_df["model"] == model_name]
    plt.plot(subset["seq_len"], subset["accuracy"], marker='o', label=model_name)

plt.xlabel("Sequence Length")
plt.ylabel("Accuracy")
plt.title("Accuracy vs Sequence Length")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

for model_name in results_df["model"].unique():
    subset = results_df[results_df["model"] == model_name]
    plt.plot(subset["seq_len"], subset["train_time_per_epoch"], marker='o', label=model_name)

plt.xlabel("Sequence Length")
plt.ylabel("Training Time per Epoch (s)")
plt.title("Training Time vs Sequence Length")
plt.legend()
plt.grid(True)
plt.show()



We train all models for sequence lengths:

- 256
- 512
- 1024

Metrics reported:

• classification accuracy  
• training time per epoch

## Results

We analyze how model accuracy changes with sequence length.

##  Analysis

### 1. Which model degrades most with longer sequences?

In general, the **standard LSTM** tends to degrade significantly as sequence length increases. Although LSTMs are designed to reduce the vanishing gradient problem using gating mechanisms, they still process tokens sequentially. As the sequence length grows (e.g., 1024 tokens), it becomes harder for the model to preserve useful information from earlier tokens.

The **Transformer** may also experience degradation when the input contains a large amount of neutral filler text. Even though the Transformer can attend to all tokens directly using self-attention, the presence of many irrelevant tokens may dilute the attention distribution and make it difficult to focus on the important parts of the sequence.

The **Linear State Space Model (SSM)** may also degrade because it uses fixed linear dynamics. Since it does not adapt its behavior based on the input content, it may struggle to capture complex dependencies in long sequences.

---

### 2. Does Selective SSM outperform LSTM or Transformer?

The **Selective SSM** generally performs better than the **Linear SSM** because it introduces an input-dependent gating mechanism:

$$
g_k = \sigma(W_g x_k)
$$

This gate allows the model to selectively control how much information passes to the output, which increases model flexibility.

Compared with **LSTM and Transformer**, the outcome depends on the experiment and dataset characteristics:

- Selective SSM often outperforms **Linear SSM**
- It may be competitive with **LSTM**
- It may still underperform **Transformer** if attention helps capture important long-range dependencies

Therefore, a safe conclusion is that **Selective SSM improves over Linear SSM and can sometimes compete with LSTM, but Transformer often remains strong for long-context modeling due to direct token-to-token interactions.**

---

### 3. Give one failure case of Transformer on long sequences

A potential failure case of the **Transformer** occurs when the input sequence contains a large amount of irrelevant neutral filler text between important tokens. In the IMDb-Long dataset, the informative parts of the review are located in the prefix $$P_1$$ and suffix $$P_3$$, while the middle section is replaced by neutral filler.

When the sequence length is very large, the attention mechanism may distribute attention weights across many irrelevant tokens. This can make it harder for the model to identify the sentiment-relevant parts of the review.

Another limitation is that the **computational complexity of self-attention grows quadratically with sequence length**:

$$
O(L^2)
$$

where \(L\) is the sequence length. As a result, Transformers become increasingly expensive in both memory and computation for very long sequences.